### Imports and R Environment Setup


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings

# Filter out the specific rpy2 environment variable warnings
warnings.filterwarnings("ignore", category=UserWarning, message='.*Environment variable ".*" redefined by R.*')

In [3]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import rpy2.robjects as ro
from rpy2.robjects import numpy2ri
from rpy2.robjects.conversion import localconverter

# Your updated package imports
from graphical_sampling.sampling import KMeansSampler
from graphical_sampling.population import Population
from package_sampling.utils import inclusion_probabilities
from graphical_sampling.index import Density 

# Initialize the converter
numpy2ri_converter = numpy2ri.converter
conv = ro.default_converter + numpy2ri_converter

# Initialize R with updated scoring functions
ro.r("""
    library(BalancedSampling)
    library(sampling)
    library(WaveSampling)

    calc_r_metrics <- function(coords, probs, sample_idx) {
        coords_mat <- as.matrix(coords)
        probs_vec <- as.numeric(probs)
        
        # Note: indices in R are 1-based. 
        # sample_idx should be adjusted before passing or inside here.
        
        # Spatial Balance (SB)
        sb_val <- tryCatch(sb(probs_vec, coords_mat, sample_idx), error = function(e) Inf)
        
        # Spatial Balance Local (SBLB)
        sblb_val <- tryCatch(sblb(probs_vec, coords_mat, sample_idx), error = function(e) Inf)
        
        # Moran's I (IB) - requires weight matrix W
        W0 <- wpik(coords_mat, probs_vec)
        W <- W0 - diag(diag(W0))
        
        # Create a binary mask for the sample
        sample_mask <- rep(0, length(probs_vec))
        sample_mask[sample_idx] <- 1
        
        ib_val <- tryCatch(IB(W, sample_mask), error = function(e) Inf)
        
        return(c(ib = ib_val, sb = sb_val, sblb = sblb_val))
    }
""")

def evaluate_sampler(sampler: KMeansSampler):
    """
    Evaluates the KMeansSampler using both internal Python properties 
    and external R metrics.
    """
    # 1. Get all possible samples and their probabilities from the joint design
    all_samples = sampler.all_samples       # Array of shape (M, n)
    all_probs = sampler.all_samples_probs   # Array of shape (M,)
    
    results = []
    
    # Use tqdm for progress tracking
    for i in tqdm(range(len(all_samples)), desc="Evaluating Samples"):
        sample_indices = all_samples[i]
        # Adjust to 1-based indexing for R
        r_sample_idx = sample_indices + 1
        
        with localconverter(conv):
            # Pass data to R
            r_metrics = ro.r['calc_r_metrics'](
                sampler.coords, 
                sampler.probs, 
                r_sample_idx
            )
            r_metrics_np = np.array(r_metrics)
        
        # Python-side Density score
        # Using the internal cached property logic
        density_score = sampler.density_scores[i]
        
        results.append({
            'sample_id': i,
            'probability': all_probs[i],
            'density': density_score,
            'moran_i': r_metrics_np[0],
            'voronoi_sb': r_metrics_np[1],
            'local_balance': r_metrics_np[2]
        })
    
    df_results = pd.DataFrame(results)
    
    # Calculate Expected Values (Weighted Averages)
    summary = {
        'Exp_Density': np.sum(df_results['density'] * df_results['probability']),
        'Exp_Moran': np.sum(df_results['moran_i'] * df_results['probability']),
        'Exp_SB': np.sum(df_results['voronoi_sb'] * df_results['probability']),
        'Exp_LocalBalance': np.sum(df_results['local_balance'] * df_results['probability'])
    }
    
    return df_results, summary



R callback write-console: Loading required package: Matrix
  


### Optimized Sampling Functions

In [4]:
import os
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from tqdm import tqdm
from rpy2.robjects import numpy2ri, pandas2ri
from rpy2.robjects.conversion import localconverter

# Define the converter context for modern rpy2
combined_converter = ro.default_converter + numpy2ri.converter + pandas2ri.converter

def run_sampling_design(method, coords, probs, n, num_samples, is_EP):
    N = len(coords)
    
    # 1. Python Methods (Nmcs and Rand)
    if method == "Nmcs":
        pop = Population(coords=coords, probs=probs)
        sampler = KMeansSampler(
            population=pop, 
            n=n, 
            n_zones=(2, 2), 
            zone_builder="sweep"
        )
        return sampler.sample(num_samples)

    if method == "Rand":
        samples_idx = np.zeros((num_samples, n), dtype=int)
        for i in range(num_samples):
            samples_idx[i] = np.random.choice(N, n, replace=False)
        return samples_idx

    # 2. R Methods (Lopi, Wave, Maxe, SCP)
    samples_idx = np.zeros((num_samples, n), dtype=int)
    
    with localconverter(combined_converter):
        ro.globalenv['coords_r'] = coords
        ro.globalenv['probs_r'] = probs
        
        # Load the core balanced sampling library
        ro.r("library(BalancedSampling)")
        
        for i in range(num_samples):
            if method == "Lopi":
                # Local Pivotal Method 2
                samples_idx[i] = np.array(ro.r("lpm2(probs_r, coords_r)")) - 1
                
            elif method == "SCP":
                # Spatially Correlated Poisson Sampling
                # scps returns 1-based indices
                samples_idx[i] = np.array(ro.r("scps(probs_r, coords_r)")) - 1
                
            elif method == "Wave":
                ro.r("library(WaveSampling)")
                mask = ro.r("wave(coords_r, probs_r)")
                samples_idx[i] = np.where(np.array(mask).astype(bool))[0]
                
            elif method == "Maxe":
                mask = ro.r("sampling::UPmaxentropy(probs_r)")
                samples_idx[i] = np.where(np.array(mask).astype(bool))[0]

    return samples_idx

In [5]:
def calculate_ht_estimator(y, sample_indices, probs):
    """
    Calculates the Horvitz-Thompson estimator for the population total.
    """
    # Filter for valid indices (stripping -1 placeholders from empty clusters)
    valid_mask = sample_indices >= 0
    valid_idx = sample_indices[valid_mask]
    
    sample_y = y[valid_idx]
    sample_probs = probs[valid_idx]
    
    # Horvitz-Thompson Total Estimate = sum(y_i / pi_i)
    return np.sum(sample_y / sample_probs)

### Metrics and Spread Calculation

In [6]:
def calculate_all_scores(coords, probs, sample_idx, n, N, density_measure, y_val):
    """
    Calculates various spatial and statistical scores for a given sample.
    
    Args:
        coords: Population coordinates.
        probs: Inclusion probabilities.
        sample_idx: 1D array of selected unit indices.
        n: Sample size.
        N: Population size.
        density_measure: An instance of the Density class.
        y_val: The variable of interest for HT estimation.
    """
    # 1. Density Score (Python)
    # The score method expects a 2D array of shape (n_samples, n)
    dens_score = density_measure.score(sample_idx.reshape(1, -1))
    
    # 2. HT Estimator Total
    # Uses the formula: Sum(y_i / pi_i)
    ht_val = np.sum(y_val[sample_idx] / probs[sample_idx])
    
    # 3. R Metrics (Spatial Balance)
    # Preparing data for the R environment
    with localconverter(conv):
        ro.globalenv['coords'] = coords
        ro.globalenv['probs'] = probs
        # R uses 1-based indexing for sample indices
        ro.globalenv['s_idx_r'] = sample_idx + 1 
        
        # Note: Your R function calc_r_metrics now takes (coords, probs, sample_idx)
        # We pass the R-adjusted 1-based indices.
        r_results = ro.r("calc_r_metrics(coords, probs, s_idx_r)")
        r_results_np = np.array(r_results)

    # Return Order: Density, Voronoi (sb), Moran (ib), Local Balance (sblb), HT_Total
    # r_results indices based on R function: c(ib, sb, sblb)
    return (
        dens_score[0],     # Density
        r_results_np[1],   # Voronoi (sb)
        r_results_np[0],   # Moran (ib)
        r_results_np[2],   # Local Balance (sblb)
        ht_val             # HT Estimator Total
    )

### The Main Execution Loop

In [7]:
# --- Main Configuration and Loop ---
warnings.filterwarnings("ignore", category=UserWarning, message='.*redefined by R.*')
folder = "/config/ws/graphical-sampling/populations"
results_folder = "data_samples"
pop_names = ["grid_uneq"]
sample_cnt = 1000 
n_size = 5

os.makedirs(results_folder, exist_ok=True)

for name in pop_names:
    # 1. Load and Prep Data
    file_path = os.path.join(folder, f"{name}_N=100.csv")
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        continue
        
    df = pd.read_csv(file_path)
    coords = df[["x", "y"]].values.astype(float)
    y_values = df["y"].values
    true_sum_y = np.sum(y_values) 

    # Calculate inclusion probabilities
    pik = inclusion_probabilities(df["prob"].values, n_size)
    N = len(pik)
    n = int(np.round(np.sum(pik)))
    is_EP = np.allclose(pik, pik[0])

    print(f"\n--- Processing {name} (N={N}, n={n}, True Total={true_sum_y:.3f}) ---")

    # 2. Setup Metrics
    pop_wrapped = Population(coords=coords, probs=pik)
    density_measure = Density(population=pop_wrapped, k=n, n_jobs=-1)

    # 3. Sampling and Scoring
    all_data = []
    methods = ["Nmcs", "Wave", "Lopi", "Scps", "Maxe", "Rand"] 

    for m in methods:
        print(f"Running {m}...")
        samples = run_sampling_design(m, coords, pik, n, sample_cnt, is_EP)
        
        for i in tqdm(range(sample_cnt), desc=f"Scoring {m}"):
            s_idx = samples[i]
            valid_s_idx = s_idx[s_idx >= 0]
            
            # Use HT for all methods, but for SRS (Rand), we will compute the correct N * mean(y_s)
            if m == "Rand":
                # SRS Estimator: N * y_bar
                est_total = N * np.mean(y_values[valid_s_idx])
                # We still need the spatial scores
                metrics = list(calculate_all_scores(coords, pik, valid_s_idx, n, N, density_measure, y_values))
                metrics[-1] = est_total # Replace HT with SRS result for Rand
            else:
                metrics = calculate_all_scores(coords, pik, valid_s_idx, n, N, density_measure, y_values)
            
            all_data.append([m] + list(metrics))

    # 4. Results Processing
    res_df = pd.DataFrame(all_data, columns=["Method", "D", "V", "M", "L", "HT"])
    
    # 5. Summary Statistics Generation
    summary = res_df.groupby("Method").agg({
        "D": ["mean", "std"], 
        "V": ["mean", "std"], 
        "M": ["mean", "std"], 
        "L": ["mean", "std"], 
        "HT": ["mean", "var"]
    })

    # Rename to requested abbreviations
    summary.columns = ["Dm", "Ds", "Vm", "Vs", "Mm", "Ms", "Lm", "Ls", "HTm", "HTv"]

    # Calculate Relative Bias (RB): (Mean_Est - TrueTotal) / TrueTotal
    summary["RB"] = (summary["HTm"] - true_sum_y) / true_sum_y

    # Calculate Efficiency (Eff) relative to SRS (Rand)
    if "Rand" in summary.index:
        rand_var = summary.loc["Rand", "HTv"]
        summary["Eff"] = rand_var / summary["HTv"].replace(0, np.nan)
    
    # Select and order requested columns
    final_cols = ["RB", "Eff", "Dm", "Ds", "Vm", "Vs", "Mm", "Ms", "Lm", "Ls"]
    summary = summary[final_cols].round(3)

    print("\nSimulation Summary:")
    print(summary)
    
    # Save the detailed and summary data
    res_df.to_csv(os.path.join(results_folder, f"samples_{name}.csv"), index=False)
    summary.to_csv(os.path.join(results_folder, f"summary_{name}.csv"))


--- Processing grid_uneq (N=100, n=5, True Total=47.368) ---


Running Nmcs...


Scoring Nmcs:   0%|          | 0/1000 [00:00<?, ?it/s]/config/ws/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "LD_LIBRARY_PATH" redefined by R and overriding existing variable. Current: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server", R: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server"
  warnings.warn(
/config/ws/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_LIBS_SITE" redefined by R and overriding existing variable. Current: "/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-l

Running Wave...


Scoring Wave: 100%|██████████| 1000/1000 [00:14<00:00, 67.07it/s]


...

Running Maxe...


Scoring Maxe: 100%|██████████| 1000/1000 [00:15<00:00, 66.57it/s]


Running Rand...


Scoring Rand: 100%|██████████| 1000/1000 [00:15<00:00, 65.42it/s]



Simulation Summary:
           RB    Eff     Dm     Ds     Vm     Vs     Mm     Ms        Lm  \
Method                                                                     
Lopi   -0.007  0.127 -0.118  0.270  0.125  0.086 -0.186  0.105     1.161   
Maxe   -0.027  0.150 -0.311  0.330  0.269  0.207 -0.017  0.139     0.717   
Nmcs   -0.019  0.136 -0.075  0.256  0.139  0.089 -0.149  0.115     0.649   
Rand   -0.001  1.000 -0.115  0.356  0.550  0.382 -0.067  0.116    34.054   
Scps   -1.000    NaN -0.964  0.000  0.000  0.000 -0.046  0.000  1133.487   
Wave   -0.011  0.129 -0.054  0.249  0.107  0.068 -0.298  0.093     0.617   

             Ls  
Method           
Lopi     16.061  
Maxe      1.360  
Nmcs      1.110  
Rand    115.438  
Scps      0.000  
Wave      1.007  


Simulation Summary:
           RB    Eff     Dm     Ds     Vm     Vs     Mm     Ms      Lm  \
Method                                                                   
Lopi    0.017  0.127 -0.100  0.276  0.129  0.082 -0.189  0.104   0.665   
Maxe    0.003  0.120 -0.311  0.318  0.273  0.213 -0.031  0.144   0.762   
Nmcs   -0.026  0.168 -0.070  0.258  0.133  0.082 -0.165  0.118   0.659   
Rand   -0.009  1.000 -0.164  0.344  0.566  0.393 -0.056  0.117  30.023   
SCP    -0.024  0.191 -0.084  0.278  0.115  0.082 -0.226  0.090   0.637   
Wave    0.014  0.151 -0.046  0.249  0.110  0.068 -0.297  0.092   0.664   

             Ls  
Method           
Lopi      1.187  
Maxe      1.459  
Nmcs      1.195  
Rand    108.511  
SCP       1.173  
Wave      1.447

# Store

In [12]:
import numpy as np
import pandas as pd
import os

# 1. Define your file list based on the image provided
# Note: I am assuming the folder name is 'data_samples' based on previous context.
# If they are in the current directory, change folder to "."
folder = "/config/ws/graphical-sampling/populations"

pop_files = {
    'clust_eq':   'clust_eq_N=100.csv',
    'clust_uneq': 'clust_uneq_N=100.csv',
    'grid_eq':    'grid_eq_N=100.csv',
    'grid_uneq':  'grid_uneq_N=100.csv',
    'random_eq':  'random_eq_N=100.csv',
    'random_uneq':'random_uneq_N=100.csv',
}

# Helper function to generate correlated variables
def generate_correlated_variable(v, correlation, seed=None):
    """Generates a new variable correlated with vector v at a specific r."""
    if seed: np.random.seed(seed)
    # 1. Create random noise
    noise = np.random.normal(0, 1, len(v))
    
    # 2. Standardize target v to remove mean/scale effects for calculation
    v_norm = (v - np.mean(v)) / np.std(v)
    
    # 3. Residualize noise (make it orthogonal to v)
    # This step ensures exact mathematical control over correlation
    noise_resid = noise - (np.dot(noise, v_norm) / np.dot(v_norm, v_norm)) * v_norm
    noise_norm = noise_resid / np.std(noise_resid)
    
    # 4. Combine to get desired correlation
    # New = r * Old + sqrt(1-r^2) * Noise
    new_var = correlation * v_norm + np.sqrt(1 - correlation**2) * noise_norm
    
    # 5. Rescale back to original range (optional, but good for probability-like vars)
    # Here we just shift it to be positive to act as a "size" variable
    new_var = new_var - np.min(new_var) + 0.1 
    return new_var

data_store = {}

print("--- Loading & Processing Data ---")
for key, fname in pop_files.items():
    path = os.path.join(folder, fname)
    
    if os.path.exists(path):
        df = pd.read_csv(path)
        
        # Basic extractions
        coords = df[['x', 'y']].values
        probs = df['prob'].values
        N = len(df)
        
        # --- LOGIC: Handle "uneq" vs "eq" files ---
        # We look for "uneq" in the key name
        if "uneq" in key:
            print(f"Processing {key}: Generating correlated auxiliaries...")
            
            # Generate the 3 auxiliary variables
            # These act as 'Target Y' variables with different correlations to inclusion probs
            y_70 = generate_correlated_variable(probs, 0.70, seed=42)
            y_80 = generate_correlated_variable(probs, 0.80, seed=43)
            y_90 = generate_correlated_variable(probs, 0.90, seed=44)
            
            # Store them so we can loop over them later
            targets_dict = {
                'y_70': y_70,
                'y_80': y_80,
                'y_90': y_90
            }
        else:
            # For Equal Probability (EP), Prob is constant. 
            # Correlation with a constant is undefined/zero. 
            # We just create one synthetic target to test spatial balance.
            print(f"Processing {key}: Standard EP file.")
            synthetic_y = (df['x'] + df['y']) + np.random.normal(0, 1, N)
            targets_dict = {'y_synthetic': synthetic_y}

        # Save to data_store
        data_store[key] = {
            'coords': coords,
            'probs': probs,
            'targets': targets_dict, # Now holds multiple Ys
            'N': N
        }
    else:
        print(f"⚠️ Warning: File {fname} not found in {folder}. Skipping.")
        
print("✅ Data Loaded Successfully.")

--- Loading & Processing Data ---
Processing clust_eq: Standard EP file.
Processing clust_uneq: Generating correlated auxiliaries...
Processing grid_eq: Standard EP file.
Processing grid_uneq: Generating correlated auxiliaries...
Processing random_eq: Standard EP file.
Processing random_uneq: Generating correlated auxiliaries...
✅ Data Loaded Successfully.


In [13]:
# Save the modified data to new CSVs
output_folder = "modified_populations"
os.makedirs(output_folder, exist_ok=True)

for key, data in data_store.items():
    # Create a DataFrame
    df = pd.DataFrame(data['coords'], columns=['x', 'y'])
    df['prob'] = data['probs']
    
    # Add the targets (y_70, y_80, etc.)
    for target_name, target_values in data['targets'].items():
        df[target_name] = target_values
        
    # Save
    df.to_csv(f"{output_folder}/{key}_modified.csv", index=False)
    print(f"Saved {key} to {output_folder}/")

Saved clust_eq to modified_populations/
Saved clust_uneq to modified_populations/
Saved grid_eq to modified_populations/
Saved grid_uneq to modified_populations/
Saved random_eq to modified_populations/
Saved random_uneq to modified_populations/


In [15]:
%load_ext rpy2.ipython

In [19]:
%%R
library(sp)
data(meuse)        # Loads the dataframe
data(meuse.grid)   # Loads the prediction grid
data(meuse.riv)    # Loads the river boundaries

# Convert to simple features (modern format)
library(sf)
meuse_sf <- st_as_sf(meuse, coords = c("x", "y"), crs = 28992)
meuse_sf

Simple feature collection with 155 features and 12 fields
Geometry type: POINT
Dimension:     XY
Bounding box:  xmin: 178605 ymin: 329714 xmax: 181390 ymax: 333611
Projected CRS: Amersfoort / RD New
First 10 features:
   cadmium copper lead zinc  elev       dist   om ffreq soil lime landuse
1     11.7     85  299 1022 7.909 0.00135803 13.6     1    1    1      Ah
2      8.6     81  277 1141 6.983 0.01222430 14.0     1    1    1      Ah
3      6.5     68  199  640 7.800 0.10302900 13.0     1    1    1      Ah
4      2.6     81  116  257 7.655 0.19009400  8.0     1    2    0      Ga
5      2.8     48  117  269 7.480 0.27709000  8.7     1    2    0      Ah
6      3.0     61  137  281 7.791 0.36406700  7.8     1    2    0      Ga
7      3.2     31  132  346 8.217 0.19009400  9.2     1    2    0      Ah
8      2.8     29  150  406 8.490 0.09215160  9.5     1    1    0      Ab
9      2.4     37  133  347 8.668 0.18461400 10.6     1    1    0      Ab
10     1.6     24   80  183 9.049 0.309702

In [20]:
%%R
# Drop geometry
meuse_df <- st_drop_geometry(meuse_sf)

# Keep only numeric columns
numeric_vars <- meuse_df[sapply(meuse_df, is.numeric)]

# Compute correlation matrix
cor_matrix <- cor(numeric_vars, use = "complete.obs")

cor_matrix


           cadmium     copper       lead       zinc       elev       dist
cadmium  1.0000000  0.9255639  0.7984435  0.9163334 -0.5651759 -0.6167057
copper   0.9255639  1.0000000  0.8166826  0.9074859 -0.5816771 -0.6112932
lead     0.7984435  0.8166826  1.0000000  0.9543124 -0.5882216 -0.5804984
zinc     0.9163334  0.9074859  0.9543124  1.0000000 -0.5970112 -0.6469027
elev    -0.5651759 -0.5816771 -0.5882216 -0.5970112  1.0000000  0.5310513
dist    -0.6167057 -0.6112932 -0.5804984 -0.6469027  0.5310513  1.0000000
om       0.7307845  0.7347169  0.5535007  0.6842578 -0.3561615 -0.5668039
dist.m  -0.6207236 -0.6160603 -0.5881192 -0.6599304  0.5092804  0.9840138
                om     dist.m
cadmium  0.7307845 -0.6207236
copper   0.7347169 -0.6160603
lead     0.5535007 -0.5881192
zinc     0.6842578 -0.6599304
elev    -0.3561615  0.5092804
dist    -0.5668039  0.9840138
om       1.0000000 -0.5890220
dist.m  -0.5890220  1.0000000


In [29]:
%%R
# Extract coordinates
coords <- st_coordinates(meuse_sf)

# Drop geometry and bind coordinates
meuse_df <- cbind(
  st_drop_geometry(meuse_sf),
  x = coords[,1],
  y = coords[,2]
)

# Convert to plain data frame (optional but safe)
meuse_df <- as.data.frame(meuse_df)

head(meuse_df)



  cadmium copper lead zinc  elev       dist   om ffreq soil lime landuse dist.m
1    11.7     85  299 1022 7.909 0.00135803 13.6     1    1    1      Ah     50
2     8.6     81  277 1141 6.983 0.01222430 14.0     1    1    1      Ah     30
3     6.5     68  199  640 7.800 0.10302900 13.0     1    1    1      Ah    150
4     2.6     81  116  257 7.655 0.19009400  8.0     1    2    0      Ga    270
5     2.8     48  117  269 7.480 0.27709000  8.7     1    2    0      Ah    380
6     3.0     61  137  281 7.791 0.36406700  7.8     1    2    0      Ga    470
       x      y
1 181072 333611
2 181025 333558
3 181165 333537
4 181298 333484
5 181307 333330
6 181390 333260


In [28]:
%%R
# Extract coordinates and drop geometry column
coords <- st_coordinates(meuse_sf)
meuse_df <- cbind(st_drop_geometry(meuse_sf), x = coords[,1], y = coords[,2])

# Save it as a CSV (in your working directory)
write.csv(meuse_df, "meuse_with_coords.csv", row.names = FALSE)


In [24]:
import os
import pandas as pd

# Load your final CSV from R
meuse_df = pd.read_csv("meuse_with_coords.csv")

# Create folder
output_folder = "modified_populations"
os.makedirs(output_folder, exist_ok=True)

# Save it
meuse_df.to_csv(f"{output_folder}/meuse_modified.csv", index=False)

print("Saved successfully.")


FileNotFoundError: [Errno 2] No such file or directory: 'meuse_with_coords.csv'